# Data Preparation - Qwen3-TTS Fine-Tuning

Converts raw recordings into `finetune/train_with_codes.jsonl`.  
Run `03_finetune.ipynb` after this.

```
finetune/raw_recordings/   ← drop WAVs here
finetune/data/             ← chunks land here (auto-created)
```

## 1. Setup

In [ ]:
import os, json, warnings, subprocess, sys, random
import torch, whisper, soundfile as sf
import IPython.display as ipd
from pathlib import Path
from pydub import AudioSegment
from pydub.silence import split_on_silence

warnings.filterwarnings("ignore")  # silence noisy library warnings (whisper/torch)

os.environ["PATH"] = "C:/ffmpeg/bin;" + os.environ["PATH"]  # pydub shells out to ffmpeg for wav export

# ── config ────────────────────────────────────────────────────────────────
RAW_DIR        = Path("finetune/raw_recordings")  # drop your WAV files here
DATA_DIR       = Path("finetune/data")            # split chunks are written here
REF_AUDIO      = Path("audio/ref_en.wav")         # voice prompt referenced by every training entry
OUT_JSONL      = Path("finetune/train_raw.jsonl")        # chunk + transcript pairs (step 6)
OUT_CODES      = Path("finetune/train_with_codes.jsonl") # same, plus tokenized audio codes (step 7)
TARGET_SR      = 24_000   # sample rate expected by the TTS model
MIN_DURATION_S = 2.0      # chunks shorter than this are too little signal to train on
MAX_DURATION_S = 15.0     # chunks longer than this likely span multiple sentences
# ──────────────────────────────────────────────────────────────────────────

DATA_DIR.mkdir(parents=True, exist_ok=True)  # create the output folder if it doesn't exist yet

## 2. Check Raw Recordings

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.wav"))
for f in raw_files:
    dur = sf.info(str(f)).duration  # reads the file header only, no full decode needed
    print(f.name, "-", round(dur / 60, 1), "min")
print("Total files:", len(raw_files))

## 3. Split into Chunks

In [ ]:
MIN_SILENCE_MS    = 600   # pause length that marks a break between utterances
SILENCE_THRESH_DB = -40   # volume below this is considered silence
KEEP_SILENCE_MS   = 100   # padding kept at chunk edges so words aren't cut off

utterances = []
for rec in raw_files:
    seg = AudioSegment.from_wav(str(rec))
    seg = seg.set_frame_rate(TARGET_SR)  # resample to match the TTS model
    seg = seg.set_channels(1)            # downmix to mono
    chunks = split_on_silence(seg, min_silence_len=MIN_SILENCE_MS,
                               silence_thresh=SILENCE_THRESH_DB,
                               keep_silence=KEEP_SILENCE_MS)
    for i, chunk in enumerate(chunks):
        dur = len(chunk) / 1000  # pydub gives length in ms, convert to seconds
        if dur < MIN_DURATION_S or dur > MAX_DURATION_S:
            continue  # skip chunks that are too short or too long for training
        out = DATA_DIR / f"{rec.stem[:30]}_{i:04d}.wav"  # truncate stem to keep filenames short
        chunk.export(str(out), format="wav")
        utterances.append((out, dur))

print("Chunks:", len(utterances))

## 4. Transcribe Chunks with Whisper large-v3

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

whisper_model = whisper.load_model("large-v3", device=device)

kept = []
for i, (wav_path, dur) in enumerate(utterances, 1):
    audio = whisper.load_audio(str(wav_path))
    result = whisper_model.transcribe(audio, language="en",
                                      beam_size=5, temperature=0.0,        # deterministic decoding for consistent labels
                                      condition_on_previous_text=False)    # chunks are unrelated, don't carry context between them
    text = result["text"].strip()
    if len(text) < 5:
        wav_path.unlink()  # transcript too short to be useful, drop the chunk too
        continue
    kept.append((wav_path, dur, text))
    if i % 200 == 0:
        print("Progress:", i, "/", len(utterances))

utterances = kept  # replace with the filtered list so later cells only see good entries
print("Kept:", len(utterances))

## 5. Review Samples (spot-check 10 random)

In [ ]:
samples = random.sample(utterances, min(10, len(utterances)))  # cap at 10 in case fewer chunks were kept
for path, dur, text in samples:
    data, sr = sf.read(str(path))
    print(path.name, "-", round(dur, 1), "s")
    print(" ", text)
    ipd.display(ipd.Audio(data, rate=sr))  # inline audio player to listen and compare against the transcript

## 6. Write train_raw.jsonl

In [ ]:
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for path, dur, text in utterances:
        f.write(json.dumps({"audio": str(path.resolve()),       # absolute path so the script runs from any cwd
                            "text": text,
                            "ref_audio": str(REF_AUDIO.resolve())},
                           ensure_ascii=False) + "\n")          # keep non-ASCII characters readable in the file

## 7. Extract Audio Codes

In [ ]:
ft_scripts = Path("finetune/scripts")
if not ft_scripts.exists():
    # sparse clone: only the finetuning/ folder of Qwen3-TTS is needed, skip the rest of the repo
    subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/QwenLM/Qwen3-TTS", str(ft_scripts)], check=True)
    subprocess.run(["git", "-C", str(ft_scripts), "sparse-checkout", "set", "finetuning"], check=True)

result = subprocess.run(
    [sys.executable, str(ft_scripts / "finetuning" / "prepare_data.py"),
     "--device",               "cuda:0",                                  # assumes a single GPU
     "--tokenizer_model_path", "Qwen/Qwen3-TTS-Tokenizer-12Hz",           # must match the base model being fine-tuned
     "--input_jsonl",          str(OUT_JSONL.resolve()),                  # entries written in the previous step
     "--output_jsonl",         str(OUT_CODES.resolve())],                 # same entries with audio codes appended
    capture_output=True, text=True,  # capture so we can print just the error tail on failure
)
if result.returncode != 0:
    print(result.stderr[-2000:])  # full traceback can be long, last lines usually have the actual error
else:
    print("Entries:", len(OUT_CODES.read_text().splitlines()))